# sets_and_dependencies / local_dependency — do the goal heads *verbalize* the edge?

**Objective (qualitative).** The dependency prompts from `clock/15` have a consumer `c` that can
only be computed from an earlier producer `b` (`c <- b`), but the instruction never names `b` — the
edge is *inferred*, not stated. Question: if we GoalScope the goal-head signal at positions around
the consumer through a **custom verbalization prompt** — *"Here I depend on …"*, *"I am waiting for
…"* — does the read-out name the producer? If it does, the goal direction carries the dependency
edge, not just "what line am I on".

No scores, no plots. Generate one prompt's code, GoalScope it through one phrasing, read the HTML by
eye. Two knobs: **which dependency prompt** and **which verbalization phrasing**.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2,3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from rich.console import Console
from rich.theme import Theme
from rich.panel import Panel
from rich.text import Text
from src.utils import logging_utils, env_utils

logging_utils.configure(level=logging.WARNING)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
RES = env_utils.DEFAULT_RESULTS_DIR

In [2]:
# ── model ──
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)
print(mt.name)

2026-08-18 15:47:34 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct


In [3]:
# ── goal heads: load the trained head mask (same recipe as clock/03_oid) ──
from src.goals.head_loc_attn import HeadMask

MASK_PATH = os.path.join(RES, f"localization/{MODEL_NAME.split('/')[-1]}/lam20/head_mask.pt")
ckpt = torch.load(MASK_PATH, map_location="cpu")
LAYERS = ckpt["layers"]
n_heads = ckpt["mask_values"].shape[1]
mask = HeadMask(len(LAYERS), n_heads)
with torch.no_grad():
    mask.values.copy_(ckpt["mask_values"])
    if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        mask.values[0:25] = 0     # keep the middle band only
        mask.values[56:] = 0

THRESHOLD = 0.9
goalscope_heads = sorted(mask.selected_heads(LAYERS, threshold=THRESHOLD))
print(f"loaded {MASK_PATH}")
print(f"layers {LAYERS[0]}–{LAYERS[-1]} · active heads (mask>{THRESHOLD}): {len(goalscope_heads)}")

loaded /disk/u/arnab/Codes/Projects/goals/results/localization/Qwen2.5-Coder-32B-Instruct/lam20/head_mask.pt
layers 0–63 · active heads (mask>0.9): 122


In [4]:
# ── a few dependency prompts, copied from clock/15 (§1) ─────────────────────
# `c <- b` means condition c can only be computed from producer b, yet the instruction NEVER names
# b — recomputing is legal, so reuse is a genuine inference. R0_control carries no edge.
from src.goals.functional import apply_chat_template

ASSIGN_RULE = (
    "For each item, in the order listed, assign the result to its own local variable, using single "
    "lowercase letters in order — `a` for the first item, `b` for the second, and so on. After all "
    "items have been computed, use a SINGLE print statement that, using those variables, reports "
    "each result. `print(f\"a: {a}, b: {b}\", ...)`"
)
MOD_RULE = ("Compute every value inside `check_conditions` (one local variable per item); do not "
            "define any helper functions.")
STUB = "def check_conditions(words: list[str]):\n"
COND_A = 'the number of words that contain the letter "e"'   # condition 1, never a dependency

def build(conds):
    body = "\n".join(f"{i+1}. {c}" for i, c in enumerate([COND_A] + conds))
    instr = ("Write a Python program that implements a function `check_conditions` which, given a "
             "list of strings words, computes the following:\n\n" + body + "\n\n"
             + ASSIGN_RULE + "\n" + MOD_RULE + "\n"
             + "Do not include any comments in your code.")
    return apply_chat_template(mt=mt, user_query=instr, incomplete_code=STUB)

PROMPTS = [
    dict(id="R0_control", dep=[],
         note="CONTROL: same ladder on b, but nothing downstream depends on it",
         conds=["the words sorted by length, longest first",
                "whether every word is entirely lowercase"]),
    dict(id="R1_sorted_len", dep=[("c", "b")],
         note="c needs the extreme of the ordering b produces",
         conds=["the words sorted by length, longest first",
                "whether the longest word is longer than four characters"]),
    dict(id="R2_lengths", dep=[("c", "b")],
         note="numeric producer; c compares two ends of the ordering",
         conds=["the lengths of the words, largest first",
                "whether the longest word is more than twice as long as the shortest word"]),
    dict(id="R4_unique", dep=[("c", "b"), ("d", "b")],
         note="two consumers could each rebuild the distinct-sorted-joined text",
         conds=["the distinct words, sorted alphabetically and joined by commas",
                "whether the distinct words, sorted alphabetically and joined by commas, "
                "would take more than twenty characters",
                "whether there is more than one distinct word"]),
    dict(id="R11_no_vowels", dep=[("c", "b")],
         note="c restates a fairly long construction — the clearest reuse-vs-rebuild case",
         conds=["each word with its vowels removed, joined by spaces",
                "whether removing every vowel from every word would leave fewer than ten "
                "characters in total"]),
]
for p in PROMPTS:
    p["prompt"] = build(p["conds"])
    p["all_conds"] = [COND_A] + p["conds"]
print(f"{len(PROMPTS)} prompts: " + ", ".join(p["id"] for p in PROMPTS))

5 prompts: R0_control, R1_sorted_len, R2_lengths, R4_unique, R11_no_vowels


## Verbalization phrasings

`apply_goalscope` injects the goal signal into a small "what are you doing?" destination prompt and
lets the model complete it. The default question is neutral; the point here is to bias the
completion toward the **dependency** with a leading question + an assistant prefill it must
continue. Add your own to `GOALSCOPE_VARIANTS` — the prefill is the phrase the read-out has to
finish, so it's where *"Here I depend on"* / *"I am waiting for"* go.

In [ ]:
# ── custom goalscope destination prompts (question + assistant prefill) ──────
def build_goalscope_prompt(mt, question, prefill=""):
    # chat-template prompt whose assistant turn is pre-seeded with `prefill` for the model to
    # continue -- same structure as goalscope._build_goalscope_prompt, but with our own wording.
    messages = [
        {"role": "system", "content": "You are a helpful assistant proficient in programming."},
        {"role": "user", "content": question},
        {"role": "assistant", "content": prefill},
    ]
    try:
        return mt.tokenizer.apply_chat_template(
            messages, add_generation_prompt=False, continue_final_message=True,
            tokenize=False, chat_template_kwargs={"enable_thinking": False})
    except TypeError:
        return mt.tokenizer.apply_chat_template(
            messages, add_generation_prompt=False, continue_final_message=True, tokenize=False)

variants: default, depend_on, waiting_for, reuse


## Run — one prompt, one phrasing

Set the two knobs and run. The cell generates the code the model actually writes for
`DEMO_PROMPT_KEY`, then GoalScopes the whole `<code>` segment through `GOALSCOPE_KEY`'s phrasing.
Read the resulting HTML by eye — hover a token to see its verbalization. The tokens to watch are
the start of the consumer line (`c = …`): does the read-out there name `b` / the producer's
construction?

In [24]:
GOALSCOPE_VARIANTS = {
    "default": dict(
        question="What are you doing at this line of the code. Keep your answer as brief as possible.",
        prefill="",
    ),
    "depend_on": dict(
        question="At this point in the code, which earlier computed value does this line depend on? Answer in a few words.",
        prefill="Here I depend on",
    ),
    "waiting_for": dict(
        question="At this point in the code, what earlier result are you waiting for (if any) before you can continue? If not waiting for anything, answer 'nothing'.",
        prefill="I am waiting for",
    ),
    "reuse": dict(
        question="At this point in the code, which earlier variable's result are you about to reuse? Answer in a few words.",
        prefill="I am about to reuse",
    ),
}

GOALSCOPE_KEY = "reuse"
gv = GOALSCOPE_VARIANTS[GOALSCOPE_KEY]
gsp = build_goalscope_prompt(mt, gv["question"], gv["prefill"])
print(gsp)

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
At this point in the code, which earlier variable's result are you about to reuse? Answer in a few words.<|im_end|>
<|im_start|>assistant
I am about to reuse


In [25]:
from src.functional import generate_with_patch
from src.goals.goalscope import apply_goalscope

# ── knobs ───────────────────────────────────────────────────────────────────
DEMO_PROMPT_KEY = "R4_unique"               # which dependency prompt (see PROMPTS above)
STRATEGY, ALPHA = "additive_norm", 0.1      # goalscope injection ("additive" | "additive_norm" | "replace")
STEER_LAYERS    = (35, 46)                  # additive strategies only
GEN_MAXNEW      = 512                       # tokens to write the full function body
SCOPE_MAXNEW    = 128                       # tokens per verbalization

p = next(x for x in PROMPTS if x["id"] == DEMO_PROMPT_KEY)
dep_s = ", ".join(f"{c}<-{pr}" for c, pr in p["dep"]) or "—"

# 1. the code the model writes for this prompt
gen = generate_with_patch(mt=mt, inputs=p["prompt"], max_new_tokens=GEN_MAXNEW,
                          do_sample=False, remove_prefix=True)[0].split("</code>")[0]
full = p["prompt"] + gen + "</code>"
rich_print(Panel(Text(gen.rstrip()),
                 title=f"[bold]{DEMO_PROMPT_KEY}[/bold]  ·  dep: {dep_s}  ·  {p['note']}",
                 title_align="left", border_style="grey42", padding=(0, 1)))

# 2. verbalize every code token through the chosen phrasing
print(f"\nphrasing [{GOALSCOPE_KEY}]  prefill={gv['prefill']!r}\n")
res = apply_goalscope(
    full, mt=mt, selected_heads=goalscope_heads,
    strategy=STRATEGY, alpha=ALPHA, steer_layer_range=STEER_LAYERS,
    batch_size=32, max_new_tokens=SCOPE_MAXNEW,
    goalscope_prompt=gsp,
    include_noncode=True,
    title=f"{DEMO_PROMPT_KEY} · {GOALSCOPE_KEY} (prefill: {gv['prefill']!r})",
)

╭─ R4_unique  ·  dep: c<-b, d<-b  ·  two consumers could each rebuild the distinct-sorted-joined text ────────────╮
│     a = sum(1 for word in words if 'e' in word)                                                                 │
│     b = ','.join(sorted(set(words)))                                                                            │
│     c = len(b) > 20                                                                                             │
│     d = len(set(words)) > 1                                                                                     │
│     print(f"a: {a}, b: {b}, c: {c}, d: {d}")                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


phrasing [reuse]  prefill='I am about to reuse'



GoalScope[additive_norm]:   0%|          | 0/3 [00:00<?, ?it/s]